<a href="https://colab.research.google.com/github/rangaraju1/AI-agents/blob/main/financial_information_chat_agent_memory_management_crewai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Objectives

- Implement Advanced CrewAI Memory: Understand how to configure and utilize CrewAI's `LongTermMemory` (with `SQLite`), `ShortTermMemory` (with RAG/Chroma), and `EntityMemory` (with RAG/Chroma) to provide agents with context persistence and entity tracking within and across sessions.

- Build Context-Aware Conversational Agents: Grasp the principles of designing agent prompts (Task description) and workflows (Crew process) that instruct the agent to effectively use its memory and tools to answer user queries in a coherent, multi-turn conversation.

# Setup

In [ ]:
!pip install -q openai==1.66.3 \
                crewai==0.114.0 \
                chromadb==0.6.3 \
                langchain==0.3.20 \
                langchain-openai==0.3.9 \
                langchain-chroma==0.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.4/567.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.5/285.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 83.0 MB/s eta 0

In [ ]:
import os
import chromadb
import sqlite3

from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import tool
from crewai.memory import LongTermMemory, ShortTermMemory, EntityMemory
from crewai.memory.storage.rag_storage import RAGStorage
from crewai.memory.storage.ltm_sqlite_storage import LTMSQLiteStorage

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from google.colab import userdata

In [ ]:
openai_api_key = userdata.get('openai_api_key')

In [ ]:
os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['OPENAI_API_BASE'] = "https://aibe.mygreatlearning.com/openai/v1"

In [ ]:
llm = LLM(
    model='gpt-4o-mini',
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    api_key=openai_api_key,
    temperature=0
)

In [ ]:
embedding_model = OpenAIEmbeddings(
    api_key=openai_api_key,
    base_url="https://aibe.mygreatlearning.com/openai/v1",
    model="text-embedding-3-small"
)

# Business Scenario

A financial services firm needs to provide its analysts and portfolio managers with rapid, accurate, and context-aware insights derived from vast amounts of regulatory filings (like SEC 10-Ks) for various companies. Analysts often ask follow-up questions or refer back to previously discussed topics during their research sessions.

Problem: Manually searching through hundreds of pages of dense 10-K documents for specific figures, risk factors, or strategic statements is time-consuming and prone to missing context from previous queries within the same research session.

Solution: Implement an agent workflow that provides an AI assistant specialized in a specific company (e.g., Tesla).
- The analyst can ask direct questions ("What were Tesla's R&D expenses in 2022 according to the 10k?"), and the agent uses the retriever tool to fetch the relevant snippet.
- The analyst can ask follow-up questions ("What risks were mentioned alongside that?", "Remind me about the CEO you mentioned earlier?"). The agent uses short-term memory to remember the immediate conversational flow and Entity Memory to recall specific details about "Tesla" or "Elon Musk" mentioned previously in this session.
- Reduces time spent manually searching documents.
- Ensures answers are grounded in the official filings when required.
- Frees up the analyst's time for higher-level analysis rather than basic information gathering.
- (Potential) Over time, the long-term memory captures how well the agent performs on certain types of queries and might offer insights (via the stored suggestions/quality) into refining the agent's instructions or identifying common areas of inquiry.

# Implementation Roadmap

**Core Idea:** For agents engaging in extended dialogues, like a research assistant answering queries over time, it's crucial to go beyond stateless responses. We need mechanisms to track the conversation's flow (what was just discussed), remember key people/places/concepts (entities), and potentially store summaries or evaluations of interactions for longer-term reference. This allows the agent to provide more relevant, coherent answers and avoid repeatedly asking for or providing the same information.

**Implementation Logic:**
1. *Foundation - External Knowledge:* First, we recognize the agent won't inherently know every detail from specific documents (Tesla 10k). We load these documents into a searchable database (ChromaDB vector store) using text embeddings. This database acts as the agent's external "filing cabinet".

2. *Access Tool:* We give the agent a "key" to this filing cabinet - a Retriever Tool. We specifically instruct the agent (via the tool's description) when to use this key: only for specific facts found in the filings, not for general knowledge.

3. *The Analyst Agent:* We create the core AI agent, defining its persona (Expert Financial Analyst), its main goal, and equipping it with the LLM brain and the Retriever Tool.

4. *The Task - Guiding the Agent:* We define the agent's instructions for each user query. Crucially, we tell it to:
    - Prioritize Memory: Check its memory first (short-term memory for recent context, Entity Memory for known entities).

    - Use Tool Selectively: If memory isn't enough and the question requires specific filing data, use the Retriever Tool.

    - Synthesize: Combine information from memory, the tool (if used), and its general knowledge to form the final answer.

5. *Configuring Memory Types:*
    - Short-Term (STM): We set up a temporary, fast-access memory (using RAG/Chroma) to store the immediate back-and-forth of the conversation (questions, answers, tool outputs) during the current run. This helps with questions like "What did I just ask?".
    - Entity (EM): We set up another temporary memory (using RAG/Chroma), specifically designed to identify and store snippets related to key entities ("Tesla", "Elon Musk", "Cybertruck") mentioned during the current run. This helps with questions like "Remind me about the CEO?".
    - Long-Term (LTM): We set up a persistent memory (using SQLite) to store a record of each task execution. After the agent provides an answer, an internal CrewAI process evaluates the interaction (based on the task description, expected output, and actual output) and saves this evaluation, including suggestions for future improvement and identified entities, to the SQLite database. This memory persists between runs.

6. *Orchestration (The Crew):* We assemble the agent, task, and configured memory components into a Crew.

7. *Simulation & Inspection:* We run a series of user questions through the crew method to simulate a conversation. Afterwards, we peek into the LTM, STM, and Entity Memory stores to observe what information was captured.

# Vector Store Setup

In [ ]:
! unzip tesla_db.zip

In [ ]:
chromadb_client = chromadb.PersistentClient(
    path='./tesla_db'
)

In [ ]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [ ]:
try:
    chromadb_client.get_collection(tesla_10k_collection)
    print(f"Collection '{tesla_10k_collection}' found.")
except Exception:
    print(f"Warning: Collection '{tesla_10k_collection}' not found in '{chromadb_client.path}'. Ensure it exists and contains data.")

In [ ]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    embedding_function=embedding_model,
    client=chromadb_client,
)

In [ ]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

The above code block sets up access to the pre-existing Chroma vector database containing Tesla 10-K document embeddings.
- `! unzip tesla_db.zip`: A shell command (specific to environments like Colab) to unzip the ChromaDB data files.
- `chromadb.PersistentClient`: Creates a client connection to the persistent ChromaDB stored on disk at ./tesla_db.
Checks if the expected collection (tesla-10k-2019-to-2023) exists within the database.
- `Chroma(...)`: Creates a LangChain Chroma vector store object, linking it to the specific collection name, the initialized `embedding_model`, and the persistent `chromadb_client`.
- `.as_retriever()`: Converts the LangChain vector store into a Retriever object, configured to find the 5 most similar documents (k=5) for a given query. This retriever will be used by the agent's tool.

# Retriever Tool Definition

In [ ]:
@tool("retriever_tool")
def retrieve_documents(query: str) -> str:
    """
    Search and return information mentioned in Tesla 10K documents from 2019 to 2023.
    Use this tool ONLY to find specific financial figures, operational details, risks,
    or other factual data directly reported in Tesla's 10-K filings.
    Do NOT use this tool for general knowledge questions about Tesla.
    """
    final_output = ''

    for result in retriever.invoke(query):
        final_output += ('\n---\n' + result.page_content + '\n---\n')

    return final_output

The above code block defines a custom tool for the CrewAI agent.

- `@tool("retriever_tool")`: Decorator that registers the retrieve_documents function as a tool named "retriever_tool" usable by CrewAI agents.

- Docstring: This is crucial. It tells the agent what the tool does and, importantly, when to use it (only for specific 10-K details, not general questions). The LLM relies heavily on this description.

- `retriever.invoke(query)`: Uses the previously created LangChain retriever to find relevant document chunks in the ChromaDB based on the query.

- The function iterates through the retrieved result objects (documents), extracts their page_content, formats them, and returns a single string containing the concatenated content.

# Agent and Task Definition

In [ ]:
tesla_analyst = Agent(
  role='Expert Financial Analyst',
  goal='Provide insightful answers to questions about Tesla, using company filings and remembering past interactions.',
  backstory=(
    "You are a seasoned financial analyst specializing in Tesla. "
    "You have access to excerpts from Tesla's 10-K filings via a retriever tool. "
    "You aim to deliver accurate, concise information, leveraging your knowledge base, the provided documents, "
    "and importantly, remembering the context of the ongoing conversation and key entities mentioned."
  ),
  verbose=True,
  llm=llm,
  tools=[retrieve_documents],
  allow_delegation=False, # This agent works alone
  # Memory will be configured in the Crew
)

The above code creates the CrewAI agent instance.
- `role`, `goal`, `backstory`: Define the agent's persona, objective, and context. The backstory explicitly mentions using the retriever tool and memory.
- `verbose=True`: Enables detailed logging of the agent's thought process and actions during execution.
- `allow_delegation=False`: Prevents this agent from assigning tasks to other agents (as it's the only one).

In [ ]:
analysis_task = Task(
  description=(
    "User Query: '{user_query}'.\n"
    "--- \n"
    "Carefully analyze the user's query.\n"
    "1. Check your existing knowledge and conversation history (memory) first.\n"
    "2. If the query asks for specific details potentially found in official filings (like financial numbers, specific risks, product details from 10k), use the retriever tool.\n"
    "3. Synthesize information from your knowledge, memory, and any retrieved documents to provide a comprehensive and accurate answer.\n"
    "4. Directly address the user's query in your final output."
  ),
  expected_output=(
    "A clear, concise, and accurate answer to the user's query, "
    "integrating information from conversation history and the retriever tool if used. "
    "Reference the use of the tool if applicable (e.g., 'According to the 10k excerpts...')."
  ),
  agent=tesla_analyst
  # context will be implicitly managed by memory
)

The above code block defines the task the agent will perform for each user query.
- `description`: Provides detailed instructions to the agent. It takes the {user_query} as input and explicitly tells the agent the order of operations: check memory, use the tool if necessary for specific filing details, synthesize, and answer. This instruction is key to making the agent use memory and tools correctly.
- `expected_output`: Describes the desired format and quality of the agent's final answer.
- `agent=tesla_analyst`: Assigns this task to the tesla_analyst agent.

# Memory Configuration

In this section we set up the storage locations and initializes the different memory components.

In [ ]:
# Define persistent storage paths (ensure these directories exist or are created)
STORAGE_BASE_DIR = "./crew_memory_storage"
LTM_DB_PATH = os.path.join(STORAGE_BASE_DIR, "long_term_memory.db")
STM_PATH = os.path.join(STORAGE_BASE_DIR, "short_term_memory")
ENTITY_PATH = os.path.join(STORAGE_BASE_DIR, "entity_memory")

In [ ]:
os.makedirs(STORAGE_BASE_DIR, exist_ok=True)
os.makedirs(STM_PATH, exist_ok=True)
os.makedirs(ENTITY_PATH, exist_ok=True)

print(f"Using LTM DB at: {LTM_DB_PATH}")
print(f"Using STM storage at: {STM_PATH}")
print(f"Using Entity storage at: {ENTITY_PATH}")

Using LTM DB at: ./crew_memory_storage/long_term_memory.db
Using STM storage at: ./crew_memory_storage/short_term_memory
Using Entity storage at: ./crew_memory_storage/entity_memory


In [ ]:
ltm = LongTermMemory(storage=LTMSQLiteStorage(db_path=LTM_DB_PATH))

In [ ]:
stm = ShortTermMemory(
    storage=RAGStorage(
        type="short_term",
        path=STM_PATH,
        embedder_config={
            "provider": "openai",
            "config": {
                "model": 'text-embedding-3-small',
                "api_key": openai_api_key,
                "api_base": "https://aibe.mygreatlearning.com/openai/v1"
            }
        }
    )
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
entity_mem = EntityMemory(
    storage=RAGStorage(
        type="entity", # Correct type flag for entity memory RAG
        path=ENTITY_PATH,
        embedder_config={
            "provider": "openai",
            "config": {
                "model": 'text-embedding-3-small',
                "api_key": openai_api_key,
                "api_base": "https://aibe.mygreatlearning.com/openai/v1"
            }
        }
    )
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


The above code block:

- Defines paths for a base storage directory and specific subdirectories/files for Long-Term Memory (SQLite DB), Short-Term Memory (ChromaDB), and Entity Memory (ChromaDB).
- `os.makedirs`: Creates these directories if they don't already exist.
- `LongTermMemory`: Initializes LTM using `LTMSQLiteStorage`, pointing it to the specified `SQLite` database file path. This memory will persist across script runs.
- `ShortTermMemory`: Initializes STM using `RAGStorage`. `type="short_term"` and `path=STM_PATH` configure it to use a ChromaDB at that path for storing recent interaction embeddings. The `embedder_config` specifies the OpenAI embedding model and API details (including the custom base URL) to use for this RAG store. This memory is typically cleared or overwritten between runs unless persistence is manually managed.
- `EntityMemory`: Initializes `Entity Memory` similarly using RAGStorage, but with `type="entity"` and its own dedicated path (ENTITY_PATH). It uses the same embedding configuration. This specifically stores information related to entities identified during the run.

# Crew Initialization

In [ ]:
crew = Crew(
  agents=[tesla_analyst],
  tasks=[analysis_task],
  process=Process.sequential,
  memory=True, # Master switch to enable memory processing
  long_term_memory=ltm,
  short_term_memory=stm,
  entity_memory=entity_mem,
  verbose=True
)

The above code assembles the final Crew.
- `agents=[tesla_analyst]`: Includes the defined agent.
- `tasks=[analysis_task]`: Includes the defined task.
- `process=Process.sequential`: Specifies that tasks should be executed one after another (necessary here as there's only one task).
- `memory=True`: The crucial flag that enables CrewAI's memory processing logic.
- `long_term_memory=ltm, short_term_memory=stm, entity_memory=entity_mem`: Passes the explicitly configured memory instances to the Crew.

# Conversation Simulation

In [ ]:
conversation_questions = [
    "What does Tesla primarily do?",
    "Who is the CEO of Tesla?",
    "Can you tell me about the main risks Tesla faces, according to their filings?", # Should trigger the tool
    "What did I just ask about?", # Should test STM
    "What are the key vehicle products besides Model S and X?", # Tests recall/synthesis
    "What are the development challenges mentioned for the Cybertruck?", # Should trigger tool & potentially relate to risk/production scaling
    "Remind me, who leads the company we are discussing?", # Should use Entity Memory for "Tesla" or "Elon Musk"
    "Summarize the main points we've discussed about Tesla's business and risks." # Tests STM/LTM synthesis
]

In [ ]:
answers = []

In [ ]:
for question in conversation_questions:
    inputs = {'user_query': question}
    # Start the crew's work
    print("\n--- Kicking off Crew ---")
    result = crew.kickoff(inputs=inputs)
    print("\n--- Crew Execution Complete ---")
    answers.append(result)

In [ ]:
for question, answer in zip(conversation_questions, answers):
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("\n---\n")

# Memory Inspection

In this section, we inspect the memory components of the crew and clarify how memory gets populated.

Under the hood, at the end of each task execution, Crew AI runs the following evaluation prompt. After the agent generates its answer (output), CrewAI internally uses the LLM with this prompt (`evaluation_query`) that includes the task details and the agent's output. The LLM's response to this evaluation prompt (containing suggestions, quality score, and extracted entities) is then processed and stored appropriately in the configured LTM (`SQLite`), STM (RAG), and Entity Memory (RAG) stores.

```python
evaluation_query = (
    f"Assess the quality of the task completed based on the description, expected output, and actual results.\n\n"
    f"Task Description:\n{task.description}\n\n"
    f"Expected Output:\n{task.expected_output}\n\n"
    f"Actual Output:\n{output}\n\n"
    "Please provide:\n"
    "- Bullet points suggestions to improve future similar tasks\n"
    "- A score from 0 to 10 evaluating on completion, quality, and overall performance"
    "- Entities extracted from the task output, if any, their type, description, and relationships"
)

```

## Inspecting long-term memory

In [ ]:
conn = sqlite3.connect(LTM_DB_PATH)

In [ ]:
cursor = conn.cursor()

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

In [ ]:
tables = cursor.fetchall()

In [ ]:
print(f"Tables found in LTM DB: {tables}")

In [ ]:
cursor.execute(f"SELECT * FROM long_term_memories")
rows = cursor.fetchall()

In [ ]:
cursor.execute(f"PRAGMA table_info(long_term_memories);")
columns_info = cursor.fetchall()
column_names = [col[1] for col in columns_info]
print(f"Columns: {column_names}")

In [ ]:
for row in rows:
    print(row)
    break

In [ ]:
for row in rows:
    print(row[1])
    print(row[2])
    print(row[3])
    print(row[4])

The above code block connects to the `long_term_memory.db` SQLite database, lists the tables, and queries the `long_term_memories` table (this is the table name CrewAI uses) to print its contents. This shows the persisted records of task executions and their evaluations.

In [ ]:
conn.close()

## Inspecting short-term and entity memory

In [ ]:
crew.short_term_memory.storage.collection.peek()

The above code block accesses the underlying Chroma collection used by the `ShortTermMemory` instance (`crew.short_term_memory.storage.collection`) and uses `.peek()` to view a sample of the stored items (representing recent conversational context).

In [ ]:
crew.entity_memory.storage.collection.peek()

Tha above code block similarly accesses the Chroma collection for `EntityMemory` (`crew.entity_memory.storage.collection`) and uses `.peek()` to view a sample of the stored items (representing identified entities and their associated context).